# Modélisation de Facteurs Latents du Risque de Crédit avec PROC HPPLS

## Résumé

Une banque de détail veut prédire trois résultats de risque de crédit corrélés — l'utilisation de carte, la trajectoire du ratio dette/revenu, et un indicateur de probabilité de défaut — à partir d'un large ensemble de prédicteurs fortement colinéaires de type bureau de crédit et macroéconomiques. La régression ordinaire s'effondre sous cette colinéarité, ce notebook utilise donc **PROC HPPLS** (moindres carrés partiels haute performance) pour extraire quelques facteurs latents qui expliquent conjointement les prédicteurs et les trois réponses, puis valide le nombre de facteurs avec un test de validation croisée de van der Voet sur un segment de portefeuille mis de côté.

## Sources de données

Toutes les données sont générées de manière synthétique dans le notebook avec `call streaminit(20240531)` — aucun fichier externe ni accès réseau.

| Jeu de données | Lignes | Variable | Rôle | Description |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Clé client unique reportée jusqu'à la sortie notée |
| | | `Segment` | Prédicteur CLASS | Segment de portefeuille : Particuliers, Aisés, Entreprises |
| | | `b1`–`b12` | Prédicteurs | 12 signaux comportementaux mensuels colinéaires de type bureau de crédit |
| | | `RatePctChg` | Prédicteur | Exposition du client au changement de taux d'intérêt |
| | | `InquiryCount` | Prédicteur | Nombre de récentes demandes de crédit (hard inquiries) |
| | | `Utilization` | Réponse | Utilisation du crédit renouvelable (%) |
| | | `DTIChange` | Réponse | Variation du ratio dette/revenu |
| | | `DefaultProp` | Réponse | Indicateur de probabilité de défaut (0–1) |
| | | `Role` | Partition | Indicateur de validation ENTRAÎNEMENT (~75 %) / TEST (~25 %) |

# Modélisation de Facteurs Latents du Risque de Crédit avec PROC HPPLS

Les banques font régulièrement face au problème du **prédicteur large et colinéaire** : des dizaines d'attributs mensuels de bureau de crédit, d'expositions macroéconomiques et de signaux comportementaux qui évoluent ensemble, utilisés pour prédire plusieurs résultats de risque eux-mêmes corrélés. Les moindres carrés ordinaires sont instables ici car la matrice de prédicteurs est quasi singulière.

Les **moindres carrés partiels (PLS)** résolvent ce problème en extrayant un petit nombre de facteurs latents à partir de la *covariance croisée* des prédicteurs (X) et des réponses (Y), de sorte que les facteurs soient ajustés pour prédire les résultats — pas seulement pour résumer X. `PROC HPPLS` est l'équivalent haute performance de `PROC PLS`, ajoutant l'exécution multithread, le partitionnement des données pour la validation, et le test de randomisation de van der Voet pour choisir le nombre de facteurs.

Dans ce notebook, nous construisons un seul modèle PLS qui prédit simultanément trois réponses corrélées de risque de crédit et utilisons un segment de validation mis de côté pour confirmer combien de facteurs latents les données supportent réellement.

## Étape 1 — Générer un panel synthétique de risque de crédit

Nous simulons 600 clients. Deux facteurs latents non observés (`stress` et `tenure`) génèrent douze signaux colinéaires de bureau de crédit `b1`–`b12` ainsi que des expositions de taux et de demandes de crédit — exactement la structure de forte corrélation pour laquelle PLS est conçu. Les trois réponses (`Utilization`, `DTIChange`, `DefaultProp`) sont des combinaisons linéaires différentes des mêmes facteurs, donc elles aussi sont corrélées. Un indicateur `Role` met de côté environ un quart du portefeuille comme segment de validation.

In [1]:
DONNÉES credit;
   APPELER streaminit(20240531);
   LONGUEUR Segment $12 Role $12;
   FAIRE CustomerID = 1 JUSQU_À 600;
      /* segment client (prédicteur catégoriel) */
      u = rand('uniform');
      SI u < 0.34 ALORS Segment = 'Particuliers';
      SINON SI u < 0.70 ALORS Segment = 'Aisés';
      SINON Segment = 'Entreprises';

      /* facteurs macro / comportementaux non observés (colinéaires) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 prédicteurs colinéaires mensuels de bureau de crédit b1-b12 */
      TABLEAU b{12} b1-b12;
      FAIRE j = 1 JUSQU_À 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      FIN;

      /* covariables macro, également liées aux facteurs */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* trois réponses corrélées de risque de crédit */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      SI DefaultProp < 0 ALORS DefaultProp = 0;

      /* met de côté ~25 % comme segment de validation */
      SI rand('uniform') < 0.25 ALORS Role = 'TEST';
      SINON Role = 'ENTRAÎNEMENT';

      SORTIE;
   FIN;
   SUPPRIMER u stress tenure j;
EXÉCUTER;



NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.37 seconds
  cpu   0.37 seconds


## Étape 2 — Ajuster le modèle PLS multi-réponses avec validation croisée

L'appel principal illustre les instructions clés de `PROC HPPLS` et les options qu'un modélisateur de risque utiliserait réellement :

- **`MODEL`** liste les trois réponses à gauche et l'ensemble complet des prédicteurs colinéaires à droite ; **`/ solution`** affiche les coefficients prédictifs finaux à l'échelle brute.
- **`CLASS Segment`** intègre le segment de portefeuille comme prédicteur catégoriel (doit précéder `MODEL`).
- **`ID CustomerID`** reporte la clé client dans la sortie notée.
- **`CVTEST(stat=press ...)`** exécute le test de randomisation de van der Voet pour choisir le nombre de facteurs de manière objective plutôt qu'à l'œil ; `seed=` le rend reproductible.
- **`PARTITION rolevar=Role(...)`** ajuste et note sur les lignes d'entraînement et met de côté le segment `TEST` afin que le PRESS de validation croisée soit mesuré hors échantillon.
- **`VARSS`** et **`DETAILS`** rapportent combien de variation X et Y chaque facteur successif explique.
- **`OUTPUT`** écrit les valeurs prédites, les résidus, les scores X, et le PRESS pour les lignes ajustées (entraînement) dans un jeu de données noté, indexé par `CustomerID`.

In [2]:
PROCÉDURE hppls DONNÉES=credit nfac=8
           cvtest(stat=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   CLASSE Segment;
   id CustomerID;
   MODÈLE Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='ENTRAÎNEMENT' test='TEST');
   SORTIE out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
   ÉTIQUETTE CustomerID="ID client" Segment="Segment"
      Utilization="Utilisation (%)" DTIChange="Var. dette/revenu"
      DefaultProp="Proba. défaut" RatePctChg="Var. taux (%)"
      InquiryCount="Nb demandes crédit";
EXÉCUTER;



The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segment: 3 levels: Aisés Entreprises Particuliers

Response Variable(s): Utilisation (%) Var. dette/revenu Proba. défaut
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Var. taux (%) Nb demandes crédit Segment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0731 74.0731     25.8294 25.8294
    2      7.9088 81.9819     37.9399 63.7693
    3      6.0977 88.0795     10.4572 74.2265
    4      1.8207 89.9003      1.4607 75.6872
    5      2.0832 91.9835      0.9925 76.6797
    6      1.3462 93.3297      0.8646 77.5443
    7      1.0034 94.3331      0.2783 77.8227
    8      0.6437 94.9768      0.1488 77.9714

Vari


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Étape 3 — Résumer le profil de risque prédit

Le modèle étant ajusté, nous dressons le profil des résultats prédits sur l'ensemble du portefeuille. `PROC MEANS` rapporte la tendance centrale et la dispersion de chaque réponse prédite afin que l'équipe risque puisse vérifier l'échelle (par exemple, l'utilisation prédite centrée autour de 40-45 %, l'indicateur de défaut proche du taux de base supposé de 8 %).

In [3]:
PROCÉDURE MOYENNES DONNÉES=scored mean std MIN MAX maxdec=3;
   VAR Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   ÉTIQUETTE Pred_Utilization="Utilisation prédite (%)" Pred_DTIChange="Variation dette/revenu prédite"
      Pred_DefaultProp="Probabilité de défaut prédite";
EXÉCUTER;


                                                  The MEANS Procedure

 Variable          Label                                      Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Utilisation prédite (%)                  47.405      10.899      29.217      82.594
 PRED_DTICHANGE    Variation dette/revenu prédite            0.649       2.503      -3.921       9.192
 PRED_DEFAULTPROP  Probabilité de défaut prédite             0.090       0.049       0.008       0.235
 -----------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Étape 4 — Examiner des clients notés individuellement

Enfin, nous listons quelques clients du segment **d'entraînement** ajusté avec leur indicateur `Role` d'origine, leur utilisation prédite, et leur résidu. (Les lignes `TEST` mises de côté ne sont volontairement pas notées, nous filtrons donc sur `Role='ENTRAÎNEMENT'` pour montrer des lignes entièrement renseignées.) C'est le type de sortie au niveau ligne qu'un analyste joindrait à un rapport de suivi de modèle ou alimenterait en aval vers un moteur de fixation de limites.

In [4]:
PROCÉDURE IMPRIMER DONNÉES=scored(obs=8) ÉTIQUETTE;
   OÙ Role = 'ENTRAÎNEMENT';
   VAR CustomerID Role Pred_Utilization Resid_Utilization;
   ÉTIQUETTE CustomerID="ID client" Role="Rôle" Pred_Utilization="Utilisation prédite (%)"
      Resid_Utilization="Résidu utilisation";
EXÉCUTER;



No observations in dataset.




NOTE: PROC PRINT data=scored



## Interprétation des résultats

Le tableau **Percent Variation Accounted for** montre que le premier facteur à lui seul absorbe environ trois quarts de la variation des prédicteurs (74,07 %, la direction colinéaire dominante « stress »), tandis que les deuxième et troisième facteurs sont ceux qui expliquent l'essentiel de la variation des *réponses* (37,94 % et 10,46 %, contre seulement 25,83 % pour le premier) — la signature de PLS réorientant les composantes vers la prédiction plutôt que vers la pure variance de X. Avec huit facteurs, les R-carrés par réponse se stabilisent à 0,81 (Utilization), 0,78 (DTIChange), et 0,74 (DefaultProp), confirmant que les trois résultats de risque de crédit sont bien captés par une structure latente de faible dimension.

Le **test de validation croisée PRESS de van der Voet** est le décideur ici : le PRESS sur le segment mis de côté chute fortement sur les quatre premiers facteurs (8816 → 4772 → 3318 → 3244) puis se stabilise et remonte légèrement, de sorte que le test sélectionne **quatre facteurs** comme modèle parcimonieux. En extraire davantage reviendrait à poursuivre le bruit dans le large bloc de bureau de crédit et à dégrader la performance hors échantillon — précisément le surajustement qu'un modèle de risque de crédit doit éviter avant le déploiement.

Les coefficients `SOLUTION` donnent à la banque une grille de notation linéaire interprétable et régularisée à l'échelle des variables d'origine, avec `RatePctChg` (≈0,80, 0,88, 0,63 selon les trois résultats) et `InquiryCount` (≈0,47, 0,36, 0,58) qui émergent comme les prédicteurs individuels les plus forts. En pratique, un modélisateur réajusterait maintenant avec `nfac=4`, surveillerait les résidus dans le jeu de données `scored`, et intégrerait les scores de facteurs ou les coefficients dans un pipeline de décision de risque en production.